In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS
workspace.lakehouse_demo.orders_incremental_practice (
    order_id STRING,
    status STRING,
    order_amount DECIMAL(12, 2),
    updated_at TIMESTAMP
)
USING DELTA
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT OVERWRITE TABLE
workspace.lakehouse_demo.orders_incremental_practice
VALUES
    ('ORD001', 'success', 100.00,
     TIMESTAMP '2026-10-04 10:00:00'),
    ('ORD002', 'failed', 200.00,
     TIMESTAMP '2026-10-04 10:00:00')
""")

display(spark.table(
    "workspace.lakehouse_demo.orders_incremental_practice"
))

order_id,status,order_amount,updated_at
ORD001,success,100.00,2026-10-04T10:00:00.000Z
ORD002,failed,200.00,2026-10-04T10:00:00.000Z


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

incoming = spark.sql("""
SELECT * FROM VALUES
    ('ORD001', 'refunded', 100.00,
     TIMESTAMP '2026-10-04 12:00:00'),
    ('ORD001', 'success', 100.00,
     TIMESTAMP '2026-10-04 11:00:00'),
    ('ORD002', 'success', 200.00,
     TIMESTAMP '2026-10-04 09:00:00'),
    ('ORD003', 'success', 300.00,
     TIMESTAMP '2026-10-04 12:00:00')
AS t(order_id, status, order_amount, updated_at)
""")

window = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc()
)

ranked = incoming.withColumn(
    "rank", F.row_number().over(window)
)

display(ranked.orderBy("order_id", "rank"))

latest = ranked.filter(F.col("rank") == 1).drop("rank")

latest.createOrReplaceTempView("incoming_orders_latest")

display(latest.orderBy("order_id"))

order_id,status,order_amount,updated_at,rank
ORD001,refunded,100.00,2026-10-04T12:00:00.000Z,1
ORD001,success,100.00,2026-10-04T11:00:00.000Z,2
ORD002,success,200.00,2026-10-04T09:00:00.000Z,1
ORD003,success,300.00,2026-10-04T12:00:00.000Z,1


order_id,status,order_amount,updated_at
ORD001,refunded,100.00,2026-10-04T12:00:00.000Z
ORD002,success,200.00,2026-10-04T09:00:00.000Z
ORD003,success,300.00,2026-10-04T12:00:00.000Z


In [0]:
spark.sql("""
MERGE INTO workspace.lakehouse_demo.orders_incremental_practice AS target
USING incoming_orders_latest AS source
ON target.order_id = source.order_id

WHEN MATCHED AND source.updated_at > target.updated_at
THEN UPDATE SET
    target.status = source.status,
    target.order_amount = source.order_amount,
    target.updated_at = source.updated_at

WHEN NOT MATCHED
THEN INSERT (order_id, status, order_amount, updated_at)
VALUES (
    source.order_id,
    source.status,
    source.order_amount,
    source.updated_at
)
""")

display(
    spark.table("workspace.lakehouse_demo.orders_incremental_practice")
    .orderBy("order_id")
)

order_id,status,order_amount,updated_at
ORD001,refunded,100.00,2026-10-04T12:00:00.000Z
ORD002,failed,200.00,2026-10-04T10:00:00.000Z
ORD003,success,300.00,2026-10-04T12:00:00.000Z


In [0]:
batch = spark.sql("""
SELECT * FROM VALUES
    ('ORD004', 'success',  400.00,
     TIMESTAMP '2026-10-04 12:00:00'),
    ('ORD004', 'refunded', 400.00,
     TIMESTAMP '2026-10-04 12:00:00'),
    ('ORD005', 'success',  500.00,
     TIMESTAMP '2026-10-04 12:00:00'),
    ('ORD005', 'success',  500.00,
     TIMESTAMP '2026-10-04 12:00:00')
AS t(order_id, status, order_amount, updated_at)
""")

In [0]:
from pyspark.sql import functions as F

# Même contenu dans toutes les colonnes : une seule copie suffit.
distinct_rows = batch.dropDuplicates()

# Après suppression des copies identiques, plusieurs lignes
# pour le même identifiant et la même date = conflit.
conflict_keys = (
    distinct_rows
    .groupBy("order_id", "updated_at")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id", "updated_at")
)

conflicts = distinct_rows.join(
    conflict_keys,
    on=["order_id", "updated_at"],
    how="inner"
)

# Politique de l'exercice : écarter toute commande
# présentant un conflit dans ce lot.
blocked_ids = conflict_keys.select("order_id").distinct()

accepted = distinct_rows.join(
    blocked_ids,
    on="order_id",
    how="left_anti"
)

display(conflicts.orderBy("order_id", "status"))
display(accepted.orderBy("order_id"))

order_id,updated_at,status,order_amount
ORD004,2026-10-04T12:00:00.000Z,refunded,400.00
ORD004,2026-10-04T12:00:00.000Z,success,400.00


order_id,status,order_amount,updated_at
ORD005,success,500.00,2026-10-04T12:00:00.000Z


In [0]:
(
    conflicts
    .withColumn("reject_reason", F.lit("Versions contradictoires à date identique"))
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.lakehouse_demo.reject_incremental_conflicts_practice"
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc()
)

accepted_latest = (
    accepted
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

accepted_latest.createOrReplaceTempView("accepted_orders_latest")

spark.sql("""
MERGE INTO workspace.lakehouse_demo.orders_incremental_practice AS target
USING accepted_orders_latest AS source
ON target.order_id = source.order_id

WHEN MATCHED AND source.updated_at > target.updated_at
THEN UPDATE SET
    target.status = source.status,
    target.order_amount = source.order_amount,
    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
INSERT (order_id, status, order_amount, updated_at)
VALUES (
    source.order_id,
    source.status,
    source.order_amount,
    source.updated_at
)
""")

display(
    spark.table("workspace.lakehouse_demo.orders_incremental_practice")
    .orderBy("order_id")
)

order_id,status,order_amount,updated_at
ORD001,refunded,100.00,2026-10-04T12:00:00.000Z
ORD002,failed,200.00,2026-10-04T10:00:00.000Z
ORD003,success,300.00,2026-10-04T12:00:00.000Z
ORD005,success,500.00,2026-10-04T12:00:00.000Z
